# Portfolio, ETF, and Engine Pipeline

One shared market state feeds pricing, portfolio analytics, benchmark risk, stress tests, ETF analytics, and one small repricing check.

## 1. Setup

Load the fixed universe and the notebook assumptions.

In [1]:
from dataclasses import replace
from decimal import Decimal

import pandas as pd
from IPython.display import display

from fuggers_py.calc import AnalyticsCurves, CurveBuilder, PricingInput, PricingRouter, PricingSpec, QuoteSide
from fuggers_py.core import Currency, Date, Frequency, Price, YearMonth, Yield
from fuggers_py.core.calendars import BusinessDayConvention
from fuggers_py.core.daycounts import DayCountConvention
from fuggers_py.market.quotes import RawQuote
from fuggers_py.market.snapshot import CurveInputs, CurvePoint, InflationFixing as RealizedCpiPoint, MarketDataSnapshot
from fuggers_py.market.sources import InMemoryInflationFixingSource as RealizedCpiSource
from fuggers_py.market.curves.funding import RepoCurve
from fuggers_py.market.curves.inflation import bootstrap_inflation_curve
from fuggers_py.portfolio import Classification, Holding, KeyRateShiftScenario, PortfolioAnalytics, PortfolioBenchmark, PortfolioBuilder, RateShockScenario, SecYieldInput, SpreadShockScenario, build_creation_basket, calculate_etf_nav_metrics, calculate_portfolio_analytics, calculate_sec_yield, estimate_yield_from_holdings, run_stress_scenarios
from fuggers_py.pricers.bonds import TipsPricer
from fuggers_py.products.bonds import TipsBond
from fuggers_py.products.rates import PayReceive, ZeroCouponInflationSwap
from fuggers_py.reference import BondReferenceData, BondType, CallScheduleEntry, FloatingRateTerms, IssuerType, Tenor
from fuggers_py.reference.bonds.types import CalendarId, CreditRating, Sector, SettlementAdjustment, YieldCalculationRules
from fuggers_py.reference.inflation import USD_CPI_U_NSA, treasury_cpi_source_from_fixings

pd.set_option('display.max_columns', 24)
pd.set_option('display.width', 180)


def dec(value: object) -> Decimal:
    return value if isinstance(value, Decimal) else Decimal(str(value))


def num(value: object | None, digits: int = 2, scale: object = 1) -> float | None:
    return None if value is None else round(float(dec(value) * dec(scale)), digits)


def next_year_month(value: YearMonth) -> YearMonth:
    if value.month == 12:
        return YearMonth(value.year + 1, 1)
    return YearMonth(value.year, value.month + 1)


def build_rules(entry: dict) -> YieldCalculationRules:
    base = getattr(YieldCalculationRules, entry['yield_rule'])()
    return replace(
        base,
        frequency=Frequency[entry['frequency']],
        calendar=CalendarId.new(entry.get('calendar', base.calendar.as_str())),
        business_day_convention=BusinessDayConvention[
            entry.get('business_day_convention', base.business_day_convention.name)
        ],
        settlement_rules=replace(
            base.settlement_rules,
            days=int(entry.get('settlement_lag_days', base.settlement_rules.days)),
            adjustment=SettlementAdjustment[
                entry.get('settlement_adjustment', base.settlement_rules.adjustment.name)
            ],
        ),
        accrual_day_count=DayCountConvention[entry.get('accrual_day_count', base.accrual_day_count.value)],
        yield_day_count=DayCountConvention[entry.get('yield_day_count', base.yield_day_count.value)],
        discount_day_count=DayCountConvention[entry.get('discount_day_count', base.discount_day_count.value)],
    )

In [2]:
cash_bond_quotes = pd.read_csv('synthetic_data/bonds/cash_bond_market_levels.csv')
nominal_curve_quotes = pd.read_csv('synthetic_data/curves/nominal_government_curve.csv')
discount_curve_quotes = pd.read_csv('synthetic_data/curves/ois_discount_curve.csv')
real_curve_quotes = pd.read_csv('synthetic_data/curves/real_rate_curve.csv')
inflation_swap_quotes = pd.read_csv('synthetic_data/curves/inflation_swap_curve.csv')
cash_breakeven_quotes = pd.read_csv('synthetic_data/curves/cash_breakeven_markers.csv')
funding_curve_quotes = pd.read_csv('synthetic_data/curves/repo_funding_curve.csv')
realized_cpi = pd.read_csv('synthetic_data/inflation/realized_cpi.csv')
reference_by_id = pd.read_csv(
    'synthetic_data/bonds/instruments.csv',
    dtype=str,
    keep_default_na=False,
).set_index('instrument_id').to_dict('index')
call_schedule_rows = pd.read_csv(
    'synthetic_data/bonds/call_schedule.csv',
    dtype=str,
    keep_default_na=False,
).to_dict('records')
positions = pd.read_csv('synthetic_data/portfolio/positions.csv').to_dict('records')

position_ids = [row['instrument_id'] for row in positions]
position_by_id = {row['instrument_id']: row for row in positions}
portfolio_id = positions[0]['portfolio_id']
etf_ids = [row['instrument_id'] for row in positions if row['basket_flag'] == 'ETF_SLICE']

as_of_date = Date.parse('2026-01-15')
settlement_date = as_of_date
quote_side = QuoteSide.MID
cpi_surprise_month = '2026-04'

government_curve_id = 'usd.gov.nominal'
discount_curve_id = 'usd.ois.discount'
real_curve_id = 'usd.tips.real'
inflation_curve_id = 'usd.cpi.zc'
funding_curve_id = 'usd.repo.gc'

liquidity_score_by_bucket = {
    'high': Decimal('0.95'),
    'medium': Decimal('0.75'),
    'low': Decimal('0.55'),
}
parallel_rate_bps = Decimal('15')
spread_bps = Decimal('20')
breakeven_bps = Decimal('10')
cpi_surprise_index_points = Decimal('0.4')
key_rate_5y_bps = Decimal('12')

shares_outstanding = dec('1000')
creation_unit_shares = dec('100')
etf_market_price = dec('118.65')
etf_liabilities = dec('250')
net_investment_income = dec('420')
gross_expense_ratio = dec('0.0018')
fee_waiver_ratio = dec('0.0004')

watch_instrument_ids = ['UST_5Y_2031', 'ACME_5Y_2031', 'FLOATCO_3M_2029']
quote_update_instrument_id = 'ACME_5Y_2031'
quote_update_clean_price = dec('100.55')
curve_update_instrument_id = 'USD_OIS_5Y'
curve_update_rate = dec('0.0368')

quote_rows = cash_bond_quotes.to_dict('records')
quote_value_by_id = {row['instrument_id']: dec(row['value']) for row in quote_rows}
quote_field_by_id = cash_bond_quotes.set_index('instrument_id')['quote_field'].to_dict()

FileNotFoundError: [Errno 2] No such file or directory: 'synthetic_data/bonds/cash_bond_market_levels.csv'

## 2. Market Objects

Build the curves, CPI history, and instruments for the book.

In [ ]:
government_curve_rows = sorted(
    nominal_curve_quotes.to_dict('records'),
    key=lambda row: Decimal(str(Tenor.parse(row['tenor']).to_years_approx())),
)
discount_curve_rows = sorted(
    discount_curve_quotes.to_dict('records'),
    key=lambda row: Decimal(str(Tenor.parse(row['tenor']).to_years_approx())),
)
real_curve_rows = sorted(
    real_curve_quotes.to_dict('records'),
    key=lambda row: Decimal(str(Tenor.parse(row['tenor']).to_years_approx())),
)
funding_curve_rows = sorted(
    funding_curve_quotes.to_dict('records'),
    key=lambda row: Decimal(str(Tenor.parse(row['tenor']).to_years_approx())),
)
cash_breakeven_curve_rows = sorted(
    cash_breakeven_quotes.to_dict('records'),
    key=lambda row: Decimal(str(Tenor.parse(row['tenor']).to_years_approx())),
)
inflation_curve_rows = sorted(
    inflation_swap_quotes.to_dict('records'),
    key=lambda row: Decimal(str(Tenor.parse(row['tenor']).to_years_approx())),
)

government_curve_input = CurveInputs.from_points(
    government_curve_id,
    settlement_date,
    [CurvePoint(Decimal(str(Tenor.parse(row['tenor']).to_years_approx())), dec(row['value'])) for row in government_curve_rows],
    source='examples',
)
discount_curve_input = CurveInputs.from_points(
    discount_curve_id,
    settlement_date,
    [CurvePoint(Decimal(str(Tenor.parse(row['tenor']).to_years_approx())), dec(row['value'])) for row in discount_curve_rows],
    source='examples',
)
real_curve_input = CurveInputs.from_points(
    real_curve_id,
    settlement_date,
    [CurvePoint(Decimal(str(Tenor.parse(row['tenor']).to_years_approx())), dec(row['value'])) for row in real_curve_rows],
    source='examples',
)
funding_curve_input = CurveInputs.from_points(
    funding_curve_id,
    settlement_date,
    [CurvePoint(Decimal(str(Tenor.parse(row['tenor']).to_years_approx())), dec(row['value'])) for row in funding_curve_rows],
    source='examples',
)
inflation_curve_input = CurveInputs.from_points(
    inflation_curve_id,
    settlement_date,
    [CurvePoint(Decimal(str(Tenor.parse(row['tenor']).to_years_approx())), dec(row['value'])) for row in inflation_curve_rows],
    source='examples',
)

curve_builder = CurveBuilder()
nominal_curve = curve_builder.add_from_inputs(government_curve_input)
discount_curve = curve_builder.add_from_inputs(discount_curve_input)
real_curve = curve_builder.add_from_inputs(real_curve_input)
funding_curve = curve_builder.add_from_inputs(funding_curve_input)
repo_curve = RepoCurve.of(funding_curve)

realized_cpi_history = [
    RealizedCpiPoint(
        'CPURNSA',
        YearMonth.parse(row['realized_month']),
        dec(row['index_value']),
        publication_date=Date.parse(row['publication_date']),
    )
    for row in realized_cpi.to_dict('records')
    if Date.parse(row['publication_date']) <= as_of_date
]
if not realized_cpi_history:
    raise ValueError('Notebook 4 requires at least one published realized CPI observation on or before the shared as_of date.')

inflation_curve = bootstrap_inflation_curve(
    [
        ZeroCouponInflationSwap.new(
            trade_date=settlement_date,
            effective_date=settlement_date,
            maturity_date=settlement_date.add_years(int(row['tenor'][:-1])),
            notional=Decimal('1000000'),
            fixed_rate=(Decimal('1') + dec(row['value'])) ** Decimal(int(row['tenor'][:-1])) - Decimal('1'),
            pay_receive=PayReceive.PAY,
            currency=Currency.USD,
            inflation_convention=USD_CPI_U_NSA,
            instrument_id=row['instrument_id'],
        )
        for row in inflation_curve_rows
    ],
    fixing_source=treasury_cpi_source_from_fixings(realized_cpi_history),
    discount_curve=discount_curve,
).curve

last_tips_maturity = max(
    Date.parse(reference_by_id[instrument_id]['maturity_date'])
    for instrument_id in position_ids
    if reference_by_id[instrument_id]['instrument_kind'] == 'tips'
)
projection_end_date = last_tips_maturity.start_of_month().add_months(1)
projection_end = YearMonth(projection_end_date.year(), projection_end_date.month())
last_observed_year_month = realized_cpi_history[-1].observation_month
cpi_surprise_year_month = YearMonth.parse(cpi_surprise_month)
if (cpi_surprise_year_month.year, cpi_surprise_year_month.month) <= (
    last_observed_year_month.year,
    last_observed_year_month.month,
):
    raise ValueError('Notebook 4 requires the CPI shock month to be projected, not already published history.')

current_month = next_year_month(last_observed_year_month)
projected_cpi_history: list[RealizedCpiPoint] = []
while (current_month.year, current_month.month) <= (projection_end.year, projection_end.month):
    projection_date = Date.parse(f'{current_month.year:04d}-{current_month.month:02d}-01').add_months(
        USD_CPI_U_NSA.observation_lag_months
    )
    projected_cpi_history.append(
        RealizedCpiPoint('CPURNSA', current_month, inflation_curve.projected_reference_cpi(projection_date))
    )
    current_month = next_year_month(current_month)

pricing_cpi_source = treasury_cpi_source_from_fixings([*realized_cpi_history, *projected_cpi_history])

instrument_map = {}
for instrument_id in position_ids:
    entry = reference_by_id[instrument_id]
    rules = build_rules(entry)

    if entry['instrument_kind'] == 'tips':
        instrument_map[instrument_id] = TipsBond.new(
            issue_date=Date.parse(entry['issue_date']),
            dated_date=Date.parse(entry['issue_date']),
            maturity_date=Date.parse(entry['maturity_date']),
            coupon_rate=dec(entry['coupon_rate']),
            inflation_convention=USD_CPI_U_NSA,
            base_reference_date=Date.parse(entry['issue_date']),
            frequency=Frequency[entry['frequency']],
            currency=Currency.from_code(entry['currency']),
            notional=dec(entry.get('notional', '100')),
            rules=rules,
            fixing_source=pricing_cpi_source,
        )
        continue

    call_schedule = ()
    if entry['instrument_kind'] == 'callable':
        call_schedule = tuple(
            CallScheduleEntry(
                exercise_date=Date.parse(row['call_date']),
                price=dec(row['call_price']),
            )
            for row in sorted(
                [row for row in call_schedule_rows if row['instrument_id'] == instrument_id],
                key=lambda row: int(row['call_order']),
            )
        )

    floating_rate_terms = None
    if entry['instrument_kind'] == 'frn':
        floating_rate_terms = FloatingRateTerms(
            index_name=entry['index'],
            spread=dec(entry['quoted_spread']),
            reset_frequency=Frequency[entry['frequency']],
            current_reference_rate=dec(entry['current_reference_rate']),
        )

    instrument_map[instrument_id] = BondReferenceData(
        instrument_id=instrument_id,
        bond_type={
            'fixed_bond': BondType.FIXED_RATE,
            'callable': BondType.CALLABLE,
            'frn': BondType.FLOATING_RATE,
        }[entry['instrument_kind']],
        issuer_type=IssuerType.SOVEREIGN if entry['issuer'] == 'US Treasury' else IssuerType.CORPORATE,
        issue_date=Date.parse(entry['issue_date']),
        maturity_date=Date.parse(entry['maturity_date']),
        currency=Currency.from_code(entry['currency']),
        notional=dec(entry.get('notional', '100')),
        coupon_rate=None if entry['instrument_kind'] == 'frn' else dec(entry['coupon_rate']),
        frequency=Frequency[entry['frequency']],
        yield_rules=rules,
        floating_rate_terms=floating_rate_terms,
        call_schedule=call_schedule,
        issuer_name=entry['issuer'],
        sector=entry['sector'],
        rating=entry['rating'],
    ).to_instrument()

market_snapshot = MarketDataSnapshot(
    as_of=settlement_date,
    quotes=tuple(
        RawQuote(
            instrument_id=instrument_id,
            value=quote_value_by_id[instrument_id],
            side=quote_side,
            as_of=settlement_date,
            currency=Currency.USD,
        )
        for instrument_id in position_ids
    ),
    curves=(
        government_curve_input,
        discount_curve_input,
        real_curve_input,
        funding_curve_input,
        inflation_curve_input,
    ),
    inflation_fixings=tuple([*realized_cpi_history, *projected_cpi_history]),
)
market_provider = market_snapshot.provider()

pricing_spec = PricingSpec(
    quote_side=quote_side,
    compute_spreads=True,
    compute_risk=True,
    compute_key_rates=True,
    compute_current_yield=True,
    include_asset_swap=True,
    route_callable_with_oas=True,
    route_floating_with_discount_margin=True,
    callable_mean_reversion=dec('0.03'),
    callable_volatility=dec('0.01'),
)
analytics_curves = AnalyticsCurves(
    discount_curve=discount_curve,
    forward_curve=discount_curve,
    government_curve=nominal_curve,
    benchmark_curve=discount_curve,
    repo_curve=repo_curve,
    inflation_curve=inflation_curve,
)

cash_breakeven_by_tenor = {row['tenor']: dec(row['value']) for row in cash_breakeven_curve_rows}
curve_snapshot = pd.DataFrame(
    [
        {
            'tenor': tenor,
            'gov_zero_%': num(nominal_curve.zero_rate(settlement_date.add_years(int(tenor[:-1]))).value(), 2, 100),
            'discount_zero_%': num(discount_curve.zero_rate(settlement_date.add_years(int(tenor[:-1]))).value(), 2, 100),
            'real_zero_%': num(None if tenor == '1Y' else real_curve.zero_rate(settlement_date.add_years(int(tenor[:-1]))).value(), 2, 100),
            'inflation_swap_%': num(
                inflation_curve.zero_inflation_rate(settlement_date, settlement_date.add_years(int(tenor[:-1]))),
                2,
                100,
            ),
            'cash_breakeven_%': num(cash_breakeven_by_tenor.get(tenor), 2, 100),
        }
        for tenor in ['1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '20Y', '30Y']
    ]
)

display(curve_snapshot)

## 3. Price The Book

Run the mixed book through the router once.

In [ ]:
router = PricingRouter()
batch_result = router.price_batch(
    [
        PricingInput(
            instrument=instrument_map[instrument_id],
            settlement_date=settlement_date,
            market_price=quote_value_by_id[instrument_id],
            pricing_spec=pricing_spec,
            curves=analytics_curves,
            market_data=market_provider,
            instrument_id=instrument_id,
        )
        for instrument_id in position_ids
    ]
)
assert not batch_result.errors, batch_result.errors
batch_outputs = batch_result.outputs

pricing_table = pd.DataFrame(
    [
        {
            'instrument_id': instrument_id,
            'sleeve': position_by_id[instrument_id]['sleeve'],
            'pricing_path': output.pricing_path,
            'price_input': f"{quote_field_by_id[instrument_id]} {num(quote_value_by_id[instrument_id], 2)}",
            'clean_price_%': num(output.clean_price, 2),
            'ytm_%': num(output.yield_to_maturity, 2, 100),
            'current_yield_%': num(output.current_yield, 2, 100),
            'z_spread_bp': num(output.z_spread, 1, 10000),
            'oas_bp': num(output.oas, 1, 10000),
            'discount_margin_bp': num(output.discount_margin, 1, 10000),
            'modified_duration': num(output.modified_duration, 3),
            'dv01_$100': num(output.dv01, 4),
        }
        for instrument_id, output in batch_outputs.items()
    ]
).sort_values(['sleeve', 'instrument_id']).reset_index(drop=True)

display(pricing_table)

## 4. Portfolio View

Roll the priced lines into portfolio and sleeve totals.

In [ ]:
portfolio_builder = PortfolioBuilder().with_currency(Currency.USD)
benchmark_builder = PortfolioBuilder().with_currency(Currency.USD)

for instrument_id in position_ids:
    row = position_by_id[instrument_id]
    entry = reference_by_id[instrument_id]
    portfolio_builder = portfolio_builder.add_holding(
        Holding(
            instrument=instrument_map[instrument_id],
            quantity=dec(row['quantity']),
            clean_price=batch_outputs[instrument_id].clean_price,
            label=instrument_id,
            classification=Classification(
                sector=Sector[entry['sector']],
                rating=CreditRating[entry['rating']],
                currency=Currency.USD,
                issuer=entry['issuer'],
            ),
            liquidity_score=liquidity_score_by_bucket.get(row['liquidity_bucket'], Decimal('0.45')),
            custom_fields={'sleeve': row['sleeve'], 'liquidity_bucket': row['liquidity_bucket']},
        )
    )
    benchmark_builder = benchmark_builder.add_holding(
        Holding(
            instrument=instrument_map[instrument_id],
            quantity=dec(row['benchmark_quantity']),
            clean_price=batch_outputs[instrument_id].clean_price,
            label=instrument_id,
            classification=Classification(
                sector=Sector[entry['sector']],
                rating=CreditRating[entry['rating']],
                currency=Currency.USD,
                issuer=entry['issuer'],
            ),
            liquidity_score=liquidity_score_by_bucket.get(row['liquidity_bucket'], Decimal('0.45')),
            custom_fields={'sleeve': row['sleeve'], 'liquidity_bucket': row['liquidity_bucket']},
        )
    )

portfolio = portfolio_builder.build()
benchmark = benchmark_builder.build()
portfolio_metrics = calculate_portfolio_analytics(portfolio, curve=discount_curve, settlement_date=settlement_date)
position_metrics = PortfolioAnalytics(portfolio).position_metrics(discount_curve, settlement_date)

sleeve_summary_raw = pd.DataFrame(
    [
        {
            'instrument_id': metric.name,
            'sleeve': position_by_id[metric.name]['sleeve'],
            'liquidity_bucket': position_by_id[metric.name]['liquidity_bucket'],
            'market_value': metric.clean_value,
            'dv01': metric.dv01,
        }
        for metric in position_metrics
    ]
)
sleeve_summary_raw = (
    sleeve_summary_raw.groupby('sleeve', as_index=False)
    .agg(
        market_value=('market_value', 'sum'),
        dv01=('dv01', 'sum'),
        liquidity_bucket=('liquidity_bucket', lambda values: values.mode().iat[0]),
        holding_count=('instrument_id', 'count'),
    )
    .sort_values('market_value', ascending=False)
    .reset_index(drop=True)
)
sleeve_summary_raw['weight'] = sleeve_summary_raw['market_value'] / portfolio_metrics.total_market_value

portfolio_view = pd.DataFrame(
    [
        {
            'segment': 'Total',
            'market_value_$': num(portfolio_metrics.total_market_value, 0),
            'weight_%': 100.0,
            'dv01_$': num(portfolio_metrics.dv01, 2),
            'modified_duration': num(portfolio_metrics.modified_duration, 3),
            'ytm_%': num(portfolio_metrics.ytm, 2, 100),
            'liquidity_bucket': '',
            'holding_count': portfolio_metrics.holding_count,
        },
        *[
            {
                'segment': row['sleeve'],
                'market_value_$': num(row['market_value'], 0),
                'weight_%': num(row['weight'], 2, 100),
                'dv01_$': num(row['dv01'], 2),
                'modified_duration': None,
                'ytm_%': None,
                'liquidity_bucket': row['liquidity_bucket'],
                'holding_count': int(row['holding_count']),
            }
            for _, row in sleeve_summary_raw.iterrows()
        ],
    ]
)

display(portfolio_view)

## 5. Benchmark

Show where the book differs from the benchmark.

In [ ]:
benchmark_pair = PortfolioBenchmark(portfolio, benchmark)
comparison = benchmark_pair.compare(discount_curve, settlement_date)

active_weights_raw = pd.DataFrame([entry.as_dict() for entry in comparison.active_weights.entries])
active_weights_raw['abs_active_weight'] = active_weights_raw['active_weight'].abs()
active_weights_raw['sector'] = active_weights_raw['name'].map(lambda instrument_id: reference_by_id[instrument_id]['sector'])
active_weights_raw = active_weights_raw.sort_values('abs_active_weight', ascending=False).reset_index(drop=True)

active_view = pd.DataFrame(
    [
        {
            'line': 'Active book',
            'sector': '',
            'portfolio_weight_%': None,
            'benchmark_weight_%': None,
            'active_weight_%': None,
            'active_dv01_$': num(comparison.active_dv01, 2),
            'active_duration': num(comparison.active_duration, 3),
            'active_z_spread_bp': num(comparison.active_z_spread, 1, 10000),
        },
        *[
            {
                'line': row['name'],
                'sector': row['sector'],
                'portfolio_weight_%': num(row['portfolio_weight'], 2, 100),
                'benchmark_weight_%': num(row['benchmark_weight'], 2, 100),
                'active_weight_%': num(row['active_weight'], 2, 100),
                'active_dv01_$': None,
                'active_duration': None,
                'active_z_spread_bp': None,
            }
            for _, row in active_weights_raw.head(8).iterrows()
        ],
    ]
)

display(active_view)

## 6. Stress

Shock rates, spreads, and CPI.

In [ ]:
stress_summary = run_stress_scenarios(
    portfolio,
    curve=discount_curve,
    settlement_date=settlement_date,
    scenarios=[
        RateShockScenario(name='parallel_rates_up', bump_bps=parallel_rate_bps),
        SpreadShockScenario(name='credit_spreads_wider', bump_bps=spread_bps),
        KeyRateShiftScenario(name='key_rate_5y_up', tenor_shocks_bps={'5Y': key_rate_5y_bps}),
    ],
)

tips_pricer = TipsPricer()
shocked_cpi_source = RealizedCpiSource(
    [
        RealizedCpiPoint(
            cpi_point.index_name,
            cpi_point.observation_month,
            cpi_point.value + cpi_surprise_index_points
            if (cpi_point.observation_month.year, cpi_point.observation_month.month)
            == (cpi_surprise_year_month.year, cpi_surprise_year_month.month)
            else cpi_point.value,
            publication_date=cpi_point.publication_date,
        )
        for cpi_point in [*realized_cpi_history, *projected_cpi_history]
    ]
)

breakeven_pnl = Decimal(0)
cpi_pnl = Decimal(0)
for instrument_id in position_ids:
    if reference_by_id[instrument_id]['instrument_kind'] != 'tips':
        continue
    quantity = dec(position_by_id[instrument_id]['quantity'])
    base_clean = batch_outputs[instrument_id].clean_price
    base_real_yield = tips_pricer.real_yield_from_clean_price(
        instrument_map[instrument_id],
        Price.new(base_clean, Currency.USD),
        settlement_date,
        fixing_source=pricing_cpi_source,
    ).ytm
    breakeven_price = tips_pricer.price_from_real_yield(
        instrument_map[instrument_id],
        Yield.new(base_real_yield.value() - breakeven_bps / Decimal(10_000), base_real_yield.compounding()),
        settlement_date,
        fixing_source=pricing_cpi_source,
    ).clean.as_percentage()
    cpi_price = tips_pricer.price_from_real_yield(
        instrument_map[instrument_id],
        base_real_yield,
        settlement_date,
        fixing_source=shocked_cpi_source,
    ).clean.as_percentage()
    breakeven_pnl += (breakeven_price - base_clean) * quantity
    cpi_pnl += (cpi_price - base_clean) * quantity

scenario_numeric = pd.DataFrame(
    [
        {'scenario': 'Parallel rates +15 bp', 'scope': 'Full portfolio', 'pnl_value': stress_summary.results['parallel_rates_up'].actual_change},
        {'scenario': 'Credit spreads +20 bp', 'scope': 'Full portfolio', 'pnl_value': stress_summary.results['credit_spreads_wider'].actual_change},
        {'scenario': 'TIPS breakeven +10 bp', 'scope': 'TIPS sleeve', 'pnl_value': breakeven_pnl},
        {'scenario': f'Projected CPI {cpi_surprise_month} +0.4 index pts', 'scope': 'TIPS sleeve', 'pnl_value': cpi_pnl},
        {'scenario': '5Y key rate +12 bp', 'scope': 'Full portfolio', 'pnl_value': stress_summary.results['key_rate_5y_up'].actual_change},
    ]
)
scenario_numeric['return_bps'] = scenario_numeric['pnl_value'] / portfolio_metrics.total_market_value * Decimal(10_000)
scenario_table = pd.DataFrame(
    [
        {
            'scenario': row['scenario'],
            'scope': row['scope'],
            'pnl_$': num(row['pnl_value'], 0),
            'return_bp_on_total_mv': num(row['return_bps'], 1),
        }
        for _, row in scenario_numeric.iterrows()
    ]
)

display(scenario_table)

## 7. ETF

Read the ETF slice as a tradable basket.

In [ ]:
etf_builder = PortfolioBuilder().with_currency(Currency.USD)
for instrument_id in etf_ids:
    row = position_by_id[instrument_id]
    entry = reference_by_id[instrument_id]
    etf_builder = etf_builder.add_holding(
        Holding(
            instrument=instrument_map[instrument_id],
            quantity=dec(row['quantity']),
            clean_price=batch_outputs[instrument_id].clean_price,
            label=instrument_id,
            classification=Classification(
                sector=Sector[entry['sector']],
                rating=CreditRating[entry['rating']],
                currency=Currency.USD,
                issuer=entry['issuer'],
            ),
            liquidity_score=liquidity_score_by_bucket.get(row['liquidity_bucket'], Decimal('0.45')),
        )
    )

etf_portfolio = etf_builder.build()
nav_metrics = calculate_etf_nav_metrics(
    etf_portfolio,
    curve=discount_curve,
    settlement_date=settlement_date,
    shares_outstanding=shares_outstanding,
    liabilities=etf_liabilities,
    market_price=etf_market_price,
)
expense_metrics = estimate_yield_from_holdings(
    etf_portfolio,
    curve=discount_curve,
    settlement_date=settlement_date,
    gross_expense_ratio=gross_expense_ratio,
    fee_waiver_ratio=fee_waiver_ratio,
)
sec_yield = calculate_sec_yield(
    SecYieldInput(
        net_investment_income=net_investment_income,
        average_shares_outstanding=shares_outstanding,
        max_offering_price=nav_metrics.nav_per_share,
        gross_expenses=expense_metrics.annual_expense_amount,
        fee_waivers=expense_metrics.net_assets * fee_waiver_ratio,
        as_of_date=settlement_date,
    )
)
basket = build_creation_basket(
    etf_portfolio,
    curve=discount_curve,
    settlement_date=settlement_date,
    shares_outstanding=shares_outstanding,
    creation_unit_shares=creation_unit_shares,
    liabilities=etf_liabilities,
)

etf_view = pd.DataFrame(
    [
        {
            'line': 'ETF summary',
            'nav_per_share_$': num(nav_metrics.nav_per_share, 2),
            'market_price_$': num(etf_market_price, 2),
            'premium_discount_bp': num(nav_metrics.premium_discount_bps, 1),
            'sec_30_day_yield_%': num(sec_yield.sec_30_day_yield, 2, 100),
            'dv01_per_share_$': num(nav_metrics.dv01_per_share, 4),
            'basket_quantity': None,
            'clean_price_%': None,
            'basket_weight_%': None,
            'sector': '',
        },
        *[
            {
                'line': component.name,
                'nav_per_share_$': None,
                'market_price_$': None,
                'premium_discount_bp': None,
                'sec_30_day_yield_%': None,
                'dv01_per_share_$': None,
                'basket_quantity': num(component.quantity, 0),
                'clean_price_%': num(batch_outputs[component.name].clean_price, 2),
                'basket_weight_%': num(component.weight, 2, 100),
                'sector': component.sector,
            }
            for component in basket.components
        ],
    ]
)

display(etf_view)

## 8. Repricing

Bump one quote and one discount-curve row, then rerun a small watch list.

In [ ]:
updated_quote_value_by_id = dict(quote_value_by_id)
updated_quote_value_by_id[quote_update_instrument_id] = quote_update_clean_price

updated_discount_curve_rows = [
    {
        **row,
        'value': str(curve_update_rate) if row['instrument_id'] == curve_update_instrument_id else row['value'],
    }
    for row in discount_curve_rows
]
updated_discount_curve_input = CurveInputs.from_points(
    discount_curve_id,
    settlement_date,
    [CurvePoint(Decimal(str(Tenor.parse(row['tenor']).to_years_approx())), dec(row['value'])) for row in updated_discount_curve_rows],
    source='examples',
)
updated_discount_curve = CurveBuilder().add_from_inputs(updated_discount_curve_input)
updated_market_snapshot = MarketDataSnapshot(
    as_of=settlement_date,
    quotes=tuple(
        RawQuote(
            instrument_id=instrument_id,
            value=updated_quote_value_by_id[instrument_id],
            side=quote_side,
            as_of=settlement_date,
            currency=Currency.USD,
        )
        for instrument_id in position_ids
    ),
    curves=(
        government_curve_input,
        updated_discount_curve_input,
        real_curve_input,
        funding_curve_input,
        inflation_curve_input,
    ),
    inflation_fixings=tuple([*realized_cpi_history, *projected_cpi_history]),
)
updated_analytics_curves = AnalyticsCurves(
    discount_curve=updated_discount_curve,
    forward_curve=updated_discount_curve,
    government_curve=nominal_curve,
    benchmark_curve=updated_discount_curve,
    repo_curve=repo_curve,
    inflation_curve=inflation_curve,
)
repriced_batch = router.price_batch(
    [
        PricingInput(
            instrument=instrument_map[instrument_id],
            settlement_date=settlement_date,
            market_price=updated_quote_value_by_id[instrument_id],
            pricing_spec=pricing_spec,
            curves=updated_analytics_curves,
            market_data=updated_market_snapshot.provider(),
            instrument_id=instrument_id,
        )
        for instrument_id in watch_instrument_ids
    ]
)
assert not repriced_batch.errors, repriced_batch.errors
repriced_outputs = repriced_batch.outputs

repricing_table = pd.DataFrame(
    [
        {
            'instrument_id': instrument_id,
            'base_clean_price_%': num(batch_outputs[instrument_id].clean_price, 2),
            'repriced_clean_price_%': num(repriced_outputs[instrument_id].clean_price, 2),
            'price_change_pts': num(repriced_outputs[instrument_id].clean_price - batch_outputs[instrument_id].clean_price, 2),
            'base_dv01_$100': num(batch_outputs[instrument_id].dv01, 4),
            'repriced_dv01_$100': num(repriced_outputs[instrument_id].dv01, 4),
            'base_z_spread_bp': num(batch_outputs[instrument_id].z_spread, 1, 10000),
            'repriced_z_spread_bp': num(repriced_outputs[instrument_id].z_spread, 1, 10000),
        }
        for instrument_id in watch_instrument_ids
    ]
)

display(repricing_table)

## 9. Takeaway

Keep the readout short.

In [ ]:
largest_sleeve = sleeve_summary_raw.iloc[0]
largest_active_position = active_weights_raw.iloc[0]
worst_scenario = scenario_numeric[scenario_numeric['scope'] == 'Full portfolio'].sort_values('pnl_value').iloc[0]

takeaway = f"""
- Largest sleeve: `{largest_sleeve['sleeve']}` at {num(largest_sleeve['weight'], 2, 100)}% of market value.
- Largest active line: `{largest_active_position['name']}` at {num(largest_active_position['active_weight'], 2, 100)}% versus benchmark.
- Worst full-book shock: `{worst_scenario['scenario']}` at ${num(worst_scenario['pnl_value'], 0)}.
"""

print(takeaway.strip())